# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [13]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [14]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [16]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'enterprise page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'learning resources page', 'url': 'https://huggingface.co/learn'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
3 days ago
•
481k
•
763
Qwen/Qwen3.5-27B
Updated
5 days ago
•
218k
•
486
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
2 days ago
•
433k
•
415
Qwen/Qwen3.5-122B-A10B
Updated
about 3 hours ago
•
127k
•
365
Qwen/Qwen3.5-397B-A17B
Updated
7 days ago
•
1.03M
•
1.16k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.76k
Qwen Image Multiple Angles 3D Camera
🎥
1.76k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
258
Omni Video Factory
🏆
258
text to video, image to video, video extend
Running
on
Zero

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n3 days ago\n•\n481k\n•\n763\nQwen/Qwen3.5-27B\nUpdated\n5 days ago\n•\n218k\n•\n486\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n2 days ago\n•\n433k\n•\n415\nQwen/Qwen3.5-122B-A10B\nUpdated\nabout 3 hours ago\n•\n127k\n•\n365\nQwen/Qwen3.5-397B-A17B\nUpdated\n7 days ago\n•\n1.03M\n•\n1.16k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.76k\nQwen Image M

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [23]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face: The AI Community Building the Future

---

## Who We Are

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. We empower ML engineers, researchers, and enthusiasts to share, explore, discover, and develop open-source models, datasets, and applications — fostering an open and ethical AI future.

Our platform serves as the central hub where millions of users worldwide create and collaborate on cutting-edge AI technology spanning modalities like text, image, video, audio, and even 3D.

---

## What We Offer

- **2M+ AI Models:** Access, share, and build upon an extensive collection of state-of-the-art machine learning models.
- **500k+ Datasets:** Explore diverse datasets to train, validate, and enhance your ML projects.
- **1M+ Applications & Spaces:** Use and create bespoke AI applications powered by models hosted in Spaces, including innovations like 3D camera angle manipulation, text-to-video generation, and custom speech synthesis.
- **Open Source Stack:** Accelerate your ML projects with our robust and transparent open-source software ecosystem.
- **Hosted Collaboration:** Seamlessly host public models, datasets, and apps with no limits on collaboration.

---

## Our Community & Customers

Our thriving, fast-growing community includes:
- Machine learning engineers and data scientists
- AI researchers and academics
- Developers building AI-powered applications
- Enterprises seeking scalable AI solutions

By providing a collaborative platform, we help the most passionate AI practitioners and organizations accelerate innovation, share knowledge, and build impressive portfolios in the advancing field of AI.

---

## Company Culture

At Hugging Face, we believe in inclusivity, openness, and ethical AI development. Our culture revolves around:
- **Collaboration:** Encouraging contributions and shared learning through an open and welcoming community.
- **Transparency:** Supporting open-source principles to advance trustworthy AI.
- **Innovation:** Empowering creativity with the latest AI tools and multimodal technologies.
- **Impact:** Driving the future of machine learning for real-world positive change.

---

## Careers & Opportunities

Join Hugging Face and be part of building the future of AI. We offer opportunities for passionate professionals eager to:
- Develop and maintain open-source ML infrastructure.
- Engage directly with a diverse, global AI community.
- Work on frontier AI projects across multiple modalities.
- Contribute to an ethical and inclusive AI ecosystem.

Explore current openings and grow your ML career with us in an environment that values innovation, diversity, and learning.

---

## Get Started with Hugging Face

- **Explore AI Applications**: Discover apps across various AI domains.
- **Browse Models & Datasets**: Find resources tailored to your ML needs.
- **Create and Collaborate**: Host your own models or datasets and build a public portfolio.
- **Accelerate with Compute**: Access paid compute resources to scale your projects.

Join our community today at [huggingface.co](https://huggingface.co) and help build the future of AI together!

---

*Hugging Face—The AI community building the future.*  
Colors: #FFD21E, #FF9D00, #6B7280  
(For brand resources, see Hugging Face Brand Assets)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## Overview

**Hugging Face** is the leading AI community dedicated to building the future of machine learning. Its platform serves as a collaborative space where machine learning engineers, data scientists, researchers, and developers come together to create, share, and advance open-source models, datasets, and applications. 

With over **2 million models** and **500,000 datasets**, Hugging Face hosts one of the largest repositories of ML resources, fostering innovation across text, image, video, audio, and even 3D modalities.

---

## What Hugging Face Offers

- **Models:** Browse and contribute to a vast collection of over 2 million machine learning models. Trending models include advanced large language models like Qwen3.5 series with billions of parameters, capable of various AI tasks such as image generation, video synthesis, and speech generation.

- **Datasets:** Access 500,000+ datasets curated for diverse AI purposes, enabling users to train and benchmark ML models efficiently.

- **Spaces:** Hosted ML applications and demos that are community-driven and open for exploration, featuring AI-powered tools like 3D camera angle changers, text-to-video converters, and custom text-to-speech generators.

- **Community Collaboration:** A thriving ecosystem where users can share work, build portfolios, and contribute to advancing the state of AI together.

- **Open Source Stack:** Accelerate machine learning projects with Hugging Face's open-source tools and infrastructure.

---

## Company Culture & Community

Hugging Face is more than a platform — it is a **vibrant AI community united by openness and collaboration**. The company embraces:

- **Open and ethical AI:** Promoting transparency, inclusiveness, and responsible development.
- **Empowerment:** Enabling the next generation of machine learning experts to learn, experiment, and grow their profiles.
- **Accessibility:** Making sophisticated AI tools and resources freely available to all.
- **Innovation:** Supporting rapid advancements across all AI modalities — from text and images to audio and 3D.

Together, Hugging Face and its global community are shaping an open-source AI future.

---

## Customers & Impact

Hugging Face’s customers and users range from independent researchers and startups to large enterprises and academic institutions. The platform serves as a backbone for:

- Building state-of-the-art AI applications
- Rapid prototyping and deploying ML models
- Collaborative research and data sharing
- Ethical and responsible AI development

Its ecosystem accelerates ML workflows and enables users to stay ahead in the fast-moving AI landscape.

---

## Careers & Opportunities

Interested in shaping the future of AI? Hugging Face offers career opportunities for passionate individuals who want to:

- Work alongside a world-class team of AI researchers, engineers, and community builders
- Contribute to open-source projects with global impact
- Foster an inclusive, collaborative, and innovative work environment
- Be part of a fast-growing company leading the AI community movement

Join Hugging Face to advance your career while helping build the next era of machine learning technology.

---

## Get Started

- Visit: [huggingface.co](https://huggingface.co)
- Explore millions of models, datasets, and applications
- Join the community and contribute to open AI projects
- Accelerate your machine learning journey with Hugging Face

---

**Hugging Face — The AI Community Building the Future**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>